# 6b. Test split, second server (optional): the same block list, walked backwards

Same as `03_score_second_server`, for the six test blocks: run it on a SECOND CPU high-RAM server right after
starting `06_test_score`. The two meet in the middle and skip each other's blocks. Not needed: `06_test_score`
alone does everything.

**Next:** `07_test_report`.

In [1]:
# --- 1. Configuration ---
PERSIST_MODE = "drive"
DRIVE_ROOT = "/content/drive/MyDrive/vggt-omega-aura-benchmark"   # where predictions, ground truth and results live.
# Work already saved there is skipped. To run EVERYTHING again from the images up, name an empty folder here,
# the same one in every notebook of the run. The Hugging Face token is still found in the usual folder's .env.
RUN_TAG = "phase7_front_medium"       # the folder of this run in persistent storage. A name from the time of the work:
                                      # every notebook of the run must use the same one, the results live under it
CAMERA = "front_medium"
MODELS = ["vggt_omega_512", "vggt_1b"]
BLOCKS = [("test", 0), ("test", 1), ("test", 2), ("test", 3), ("test", 11), ("test", 12)]
# The six test blocks that can be downloaded at the pinned dataset revision: 107 scenes. Nothing was tuned on them.
LIDAR_POLICY = "ouster_only"          # same six sensors in every scene, so ground-truth density is comparable
VALIDATE_DOWNLOAD = True              # run the dataset toolkit's own validator on each new block
REVERSE = True                        # backwards here; `06_test_score` walks the same list forwards,
                                      # so the two servers meet in the middle. Everything else must equal `06_test_score`.
WORKERS = None                        # scenes scored at once. None = one per CPU core (max 8). The numbers do not depend on it

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Start the session ---
from vggt_aura.session import start_session

# build_cpp=True compiles the C++ geometry core on this server (about 15 s). Ground truth is then built
# with it, which gives exactly the same result as the Python reference, faster. If the build fails,
# everything still runs, in Python.
session = start_session(persist_mode=PERSIST_MODE, drive_root=DRIVE_ROOT, require_gpu=False, build_cpp=True)

Mounted at /content/drive
installing vggt_omega
installing pybind11
installing fzi_aura
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)
$ cmake -S /content/vggt-omega-aura-benchmark/cpp -B /content/vggt-omega-aura-benchmark/cpp/build -DCMAKE_BUILD_TYPE=Release -Dpybind11_DIR=/usr/local/lib/python3.13/dist-packages/pybind11/share/cmake/pybind11 -DPython_EXECUTABLE=/usr/bin/python3
$ cmake --build /content/vggt-omega-aura-benchmark/cpp/build --config Release -j
C++ core    : built


In [4]:
# --- 4. Process the blocks ---
import pandas as pd
from vggt_aura import aura_data as ad, pipeline as pl

pd.set_option("display.width", 220)
chunks, scene_blocks, hub_files = ad.fetch_release_tables(session.data_root / "_release_tables")
EXCLUDED = ad.fetch_excluded_scene_ids(session.data_root / "_release_tables")   # faulty scenes the maintainers exclude
print("scenes excluded by the dataset:", len(EXCLUDED))
available = ad.available_blocks(chunks, scene_blocks, hub_files, [pl.CAMERA_LAYER, pl.LIDAR_LAYER])
import uuid
ME = ("backward-" if REVERSE else "forward-") + uuid.uuid4().hex[:6]      # this server's name on its claims
summaries, left_to_the_other = [], []
for split, block in (list(reversed(BLOCKS)) if REVERSE else list(BLOCKS)):
    if not pl.block_is_done(session.persist_root, RUN_TAG, MODELS, split, block) \
            and not pl.claim_block(session.persist_root, RUN_TAG, split, block, ME, max_age_s=1500):
        print(f"=== {pl.block_tag(split, block)}: the other server is on it, skipped ===")
        left_to_the_other.append((split, block))
        continue
    assert (split, block) in available.index, f"block {(split, block)} is not downloadable with camera + LiDAR"
    print(f"=== {pl.block_tag(split, block)} ({available.loc[(split, block), 'total_gb']} GB) ===")
    try:
        summary = pl.process_block(session, split, block, ad.block_scene_ids(scene_blocks, split, block, EXCLUDED), CAMERA, MODELS,
                                   RUN_TAG, lidar_policy=LIDAR_POLICY, validate=VALIDATE_DOWNLOAD,
                                   scene_names=ad.block_scene_names(scene_blocks, split, block, EXCLUDED), workers=WORKERS)
    finally:                 # a claim must not outlive a crash: a re-run gets a new name and would wait for it
        pl.release_claim(session.persist_root, RUN_TAG, split, block, ME)
    print(" ", summary)
    summaries.append(summary)
print()
print(pd.DataFrame([{k: v for k, v in s.items() if k not in ("sensors", "validation")} for s in summaries]).to_string(index=False))
still_open = [b for b in left_to_the_other if not pl.block_is_done(session.persist_root, RUN_TAG, MODELS, *b)]
if still_open:
    print()
    print("left to the other server and not finished yet:", still_open, "| if that server stopped, run this notebook again")

scenes excluded by the dataset: 8
=== test_block000012 (3.02 GB) ===
  downloading with the toolkit, decompressing with xz on all 8 cores


  fast unpack: {'archives': 3, 'xz_decompressed_on_all_cores': 1, 'download_s': 19.0, 'verify_and_decompress_s': 17.7, 'extract_s': 12.6}
  predictions: 0 made now, the rest loaded (0 s) | scoring 7 scenes with 7 worker(s)
  2025-06-12-14-23-14|400     30.6 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-06-13-09-11-24|112     26.7 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|199     32.8 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|180     33.9 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-12-58-34|1       32.9 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|133     28.2 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|209     44.2 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  validator: {'ok': True, 'scenes_checked': 7, 'errors': []}
  

  fast unpack: {'archives': 3, 'xz_decompressed_on_all_cores': 1, 'download_s': 30.6, 'verify_and_decompress_s': 67.3, 'extract_s': 41.9}
  predictions: 0 made now, the rest loaded (1 s) | scoring 20 scenes with 8 worker(s)
  2026-06-03-11-25-54|3       30.2 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-17-05-20|111     27.9 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-03-10-44-05|98      31.9 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-18-10-56-11|33      25.4 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-02-17-05-20|113     30.4 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|208     65.2 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|207     58.2 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|210     58.5 s | vggt_omega_512: truth b

  fast unpack: {'archives': 3, 'xz_decompressed_on_all_cores': 1, 'download_s': 26.9, 'verify_and_decompress_s': 56.9, 'extract_s': 39.9}
  predictions: 0 made now, the rest loaded (1 s) | scoring 20 scenes with 8 worker(s)
  2025-08-04-11-19-58|85      53.7 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|39      55.1 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|34      51.9 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|84      46.4 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|38      50.7 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|52      53.6 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-18-10-56-11|24      42.9 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2026-06-18-10-37-06|52      60.2 s | vggt_omega_512: truth b

  fast unpack: {'archives': 3, 'xz_decompressed_on_all_cores': 1, 'download_s': 27.4, 'verify_and_decompress_s': 64.1, 'extract_s': 46.3}
  predictions: 0 made now, the rest loaded (1 s) | scoring 20 scenes with 8 worker(s)
  2025-08-04-11-19-58|95      61.2 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|215     56.1 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|35      62.0 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|41      56.4 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|214     60.1 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|67      65.3 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|42      53.1 s | vggt_omega_512: truth built (cpp) | vggt_1b: truth built (cpp)
  2025-08-04-11-19-58|40      61.8 s | vggt_omega_512: truth b

In [5]:
# --- 5. What the run holds so far ---
for model in MODELS:
    rows, scenes = pl.load_run(session.persist_root, RUN_TAG, model)
    print(f"{model}: {scenes['scene_id'].nunique() if len(scenes) else 0} scenes in "
          f"{scenes[['split', 'block']].drop_duplicates().shape[0] if len(scenes) else 0} blocks")

vggt_omega_512: 394 scenes in 21 blocks
vggt_1b: 394 scenes in 21 blocks
